# Plataformas de herramientas: Composio 

Hasta ahora escribíamos cada herramienta **a mano** (una función para el tiempo, otra para divisas…). Funciona, pero imagina que quieres que tu agente use **Gmail, GitHub, Slack, Notion, Google Calendar**… Tendrías que programar cada integración **y** gestionar el inicio de sesión (login) de cada servicio. Un montón de trabajo.

**Composio** resuelve eso: es una **plataforma con más de 1.000 herramientas ya hechas** para apps populares, y además **gestiona la autenticación** (el login con OAuth) por ti. Tú solo pides las herramientas que quieres y las conectas a tu agente.

## Instalación y claves

Necesitas dos claves (ambas gratuitas):
- La de tu **modelo** (Groq u OpenRouter), como siempre.
- La de **Composio**: créala en **https://composio.dev** → *Settings* → *API Keys* (empieza por algo tipo `comp_...`).

In [ ]:
#!pip install -q composio composio_openai

In [37]:
from openai import OpenAI
from getpass import getpass
import json

API_KEY = getpass("Pega tu clave de Groq u OpenRouter: ")

BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "qwen/qwen3.6-27b"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [3]:
from composio import Composio
from composio_openai import OpenAIProvider


COMPOSIO_KEY = getpass("Pega tu clave de Composio: ")


USER_ID = "Diego"

composio = Composio(provider=OpenAIProvider(), api_key=COMPOSIO_KEY)


c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


## La idea: *toolkits* (colecciones de herramientas)

En Composio, las herramientas se agrupan por app en **toolkits**: `GITHUB`, `GMAIL`, `SLACK`, `NOTION`, `HACKERNEWS`, etc. Pides el que necesites y te devuelve sus herramientas **ya en el formato** que entiende el function calling.

Vamos a pedir el toolkit de **HackerNews** (la web de noticias de tecnología).

In [31]:
tools = composio.tools.get(user_id=USER_ID, tools=["HACKERNEWS_GET_TOP_STORIES"])

print("Número de herramientas recibidas:", len(tools))
print("\nAlgunas de ellas:")
for t in tools:
    print("  -", t["function"]["name"])

Número de herramientas recibidas: 1

Algunas de ellas:
  - HACKERNEWS_GET_TOP_STORIES


In [32]:
tools

[{'function': {'name': 'HACKERNEWS_GET_TOP_STORIES',
   'description': 'Get up to 500 top story IDs from HackerNews including jobs. Returns story IDs sorted by front page position. Use the returned IDs with get_item_with_id action to fetch full story details.',
   'parameters': {'type': 'object',
    'title': 'GetTopStoriesRequest',
    'properties': {'print': {'enum': ['pretty'],
      'type': 'string',
      'title': 'PrintFormat',
      'description': 'Output formatting options for the API response. Please provide a value of type string.'}}},
   'strict': None},
  'type': 'function'}]

Fíjate: **no hemos programado ninguna** de esas herramientas. Composio nos las da hechas. Cada una tiene su `name`, `description` y `parameters`, exactamente como las fichas que escribíamos a mano en el Notebook 1.

## Una llamada: el modelo decide, Composio ejecuta

El flujo tiene un matiz nuevo respecto a antes: cuando el modelo pide una herramienta, **Composio la ejecuta por nosotros** con `composio.provider.handle_tool_calls(...)`. Veámoslo paso a paso.

In [33]:
for tool in tools:
    if "function" in tool:
        # Si el parámetro strict existe y es None, lo borramos
        if "strict" in tool["function"] and tool["function"]["strict"] is None:
            del tool["function"]["strict"]

In [34]:
print(len(tools))

1


In [41]:
pregunta = "¿Cuáles son las noticias más populares? Dame 3 titulares."

mensajes = [
    {"role": "system", "content": "Eres un asistente que usa herramientas para responder."},
    {"role": "user", "content": pregunta},
]

respuesta = cliente.chat.completions.create(
    model=MODELO, messages=mensajes, tools=tools, tool_choice="auto",
)
msg = respuesta.choices[0].message

if msg.tool_calls:
    print("El modelo quiere usar:", msg.tool_calls[0].function.name)
    resultado = composio.provider.handle_tool_calls(response=respuesta, user_id=USER_ID)
    print("Composio ha ejecutado la herramienta y ha devuelto datos.")
else:
    print("El modelo respondió sin herramientas:", msg.content)

El modelo quiere usar: HACKERNEWS_GET_TOP_STORIES
Composio ha ejecutado la herramienta y ha devuelto datos.


Ese `resultado` contiene los datos reales de HackerNews. Ahora se los devolvemos al modelo (mensaje `tool`) para que redacte la respuesta final, igual que en el function calling normal.

In [42]:
mensajes.append(msg)  
mensajes.append({
    "role": "tool",
    "tool_call_id": msg.tool_calls[0].id,
    "content": str(resultado),
})

final = cliente.chat.completions.create(model=MODELO, messages=mensajes, tools=tools)
print(final.choices[0].message.content)

He recuperado los identificadores de las historias más vistas actualmente en **HackerNews** (una de las principales comunidades de tecnología y startups). Los 3 primeros ID, que corresponden a los titulares más populares del momento, son:

1. `48713832`
2. `48709670`
3. `48708644`

⚠️ **Nota:** En este entorno no tengo acceso a la herramienta que transforma estos ID en los títulos completos. Para leer los titulares exactos y sus enlaces, puedes visitar directamente la página principal de [HackerNews](https://news.ycombinator.com) o usar servicios como la [API oficial](https://github.com/HackerNews/API) que permiten consultar `https://hacker-news.firebaseio.com/v0/item/<ID>.json`.

¿Te gustaría que te ayude a buscar noticias de otro sector, a analizar algún tema específico o a formatear estos datos de otra manera?


## ¿Y las apps con login? (Gmail, GitHub…)

HackerNews es público, pero la mayoría de apps (Gmail, GitHub, Slack…) requieren **tu permiso** para acceder a tu cuenta. Aquí es donde Composio brilla: **gestiona el login (OAuth) por ti**.

El flujo, a grandes rasgos, es:
1. Pides conectar la app: Composio te da un **enlace**.
2. Abres el enlace y autorizas (una sola vez).
3. Composio **guarda el permiso** asociado a tu `user_id` y lo usa automáticamente en cada herramienta.

En código se parece a esto (no lo ejecutamos aquí para no liarnos con permisos):

```python
# Iniciar la conexión con Gmail para este usuario
conexion = composio.connected_accounts.initiate(user_id=USER_ID, toolkit="GMAIL")
print("Abre este enlace para autorizar:", conexion.redirect_url)
# ...tras autorizar, ya puedes pedir y usar las herramientas de GMAIL con ese user_id
```